# Routing capacity and Gaussian source-norm compensation

Purpose: define four encoders for a bounded full-JEPA screen. Inputs: images or valid patch tokens and original spatial indices. Outputs: original token shapes, both online and EMA encoders, the inherited predictor and JEPA objective. Definitions import in seconds and never start training. Run from the consumer Notebook14; its runner owns checkpoints under `checkpoints/<run_id>/`, the single writer, and exact resume. This notebook has no mutable experiment state.

The `normalized_kernel` control preserves Notebook13's six-coordinate, per-band-normalized interaction exactly. The corrected variants remove the semantic source-norm bias in each band while retaining anchored 2D spatial Gaussian preferences:
$$P^{(r)}_{ij}=\operatorname{softmax}_j\left[-\frac{\|p_i-p_j\|^2}{2\ell_r^2}+\frac{q_i^\top k_j}{\ell_r^2 f^2}\right],\qquad A_{ij}=\sum_r\pi_r P^{(r)}_{ij}.$$
This follows by adding $\|k_j\|^2/(2\ell_r^2f^2)$ to a semantic Gaussian and dropping its exact query-row constant. Query and source maps start equal with separate trainable storage; this is a directed interaction, not a proved symmetric manifold metric.

The corrected narrow condition keeps six semantic coordinates/head and $f=1$. The wider condition uses32 coordinates/head, copying common first6 coordinates and retaining26 independent extra coordinates with the same initialization variance. Its default fixed length factor $f=(32/6)^{1/4}$ scales temperature by $\sqrt{32/6}$; a frozen training-only entropy calibration may override this factor. Spatial bandwidths remain unchanged and trainable; the factor itself is fixed. Extra active parameters are reported. Values remain64 dimensions/head for the actual width192/head3 model. All computations remain exact dense and quadratic; no FMM acceleration is claimed.

An isolated raw64D single-Gaussian identity helper verifies exact standard attention equivalence with the same Q/K/V, masks and $\sqrt{64}$ temperature. It is not another training arm. Only tagged `ahr-export` cells may be imported; upstream training cells are never executed.

In [ ]:
"""Dense routing-capacity diagnostic encoders; no FMM approximation or speed claim."""
from pathlib import Path
import copy, importlib.util, math, sys
import torch
from torch import nn
from torch.nn import functional as F
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.models import JEPA, Predictor
_ROOT=Path(__file__).resolve().parents[2]
if 'fmm13_encoder' not in sys.modules:
    _path=_ROOT/'experiments/13_product_geometry/encoder.ipynb'
    _spec=importlib.util.spec_from_loader('fmm13_encoder',TaggedNotebookLoader(_path))
    _module=importlib.util.module_from_spec(_spec);sys.modules['fmm13_encoder']=_module
    try:_spec.loader.exec_module(_module)
    except BaseException:sys.modules.pop('fmm13_encoder',None);raise
PREVIOUS=sys.modules['fmm13_encoder'];OLD=PREVIOUS.OLD
BACKEND='exact-dense-source-norm-compensated-routing-v1'
VARIANTS=('vit','normalized_kernel','compensated_kernel','compensated_wide_kernel')

def default_content_length_factor(semantic_dim):
    """ell^2 scales by sqrt(D/6), preserving iid dot-product score variance."""
    if int(semantic_dim)<1:raise ValueError('Positive semantic dimension required')
    return (int(semantic_dim)/6.)**.25

class RoutingMixer(OLD.NormalizedRBFMixer):
    """Compensate semantic source norm independently in each normalized band.
    The row-constant query norm is also dropped exactly before row softmax.
    Spatial distances remain Gaussian; values retain full dim/heads per head.
    """
    def __init__(self,dim,heads=3,variant='compensated_kernel',semantic_dim=None,
                 bandwidths=(.15,.35,.75),content_scale=.5,content_length_factor=None):
        if variant not in VARIANTS[2:]:raise ValueError('Expected a compensated variant')
        expected=32 if variant=='compensated_wide_kernel' else 6
        semantic_dim=expected if semantic_dim is None else int(semantic_dim)
        if semantic_dim!=expected:raise ValueError('Registered compensated dimensions are6and32')
        factor=default_content_length_factor(semantic_dim) if content_length_factor is None else float(content_length_factor)
        if not math.isfinite(factor) or factor<=0:raise ValueError('Positive finite fixed content_length_factor required')
        super().__init__(dim,heads,'learned_kernel',semantic_dim+2,bandwidths,content_scale)
        self.variant=variant;self.semantic_dim=semantic_dim;self.content_length_factor=factor;self.views=1
        self.backend='exact dense per-band semantic source-norm compensated routing; quadratic token pairs'

    def _band_log_kernels(self,query,source):
        dtype=query.dtype
        ellp=self.log_bandwidths.to(dtype).clamp(math.log(.03),math.log(2.)).exp()
        ellc=ellp*self.content_length_factor
        dp=PREVIOUS._squared_distance(query[...,:2],source[...,:2])
        similarity=query[...,2:]@source[...,2:].transpose(-1,-2)
        return torch.stack([-dp/(2*ellp[None,:,r,None,None].square())+
            similarity/ellc[None,:,r,None,None].square() for r in range(ellp.shape[-1])],dim=2)

    def _log_kernel(self,query,source):
        bands=self._band_log_kernels(query,source)
        return torch.logsumexp(bands+self.mixture_logits.to(query.dtype).log_softmax(-1)[None,:,:,None,None],dim=2)

    def kernel_state(self,x,indices,valid,grid):
        if not bool(valid.any(-1).all()):raise ValueError('Each sample requires a valid source')
        with torch.autocast(device_type=x.device.type,enabled=False):
            query,source=self._coordinates(x,indices,valid,grid);bands=self._band_log_kernels(query,source)
            logits=bands.masked_fill(~valid[:,None,None,None,:],float('-inf'))
            per_band=logits.softmax(-1)
            pi=self.mixture_logits.to(query.dtype).softmax(-1)[None,:,:,None,None]
            weights=(pi*per_band).sum(2)
            weights=torch.where(valid[:,None,:,None],weights,torch.zeros_like(weights))
        return {'query_coordinates':query,'source_coordinates':source,'query_views':[query],'source_views':[source],
            'weights':weights,'log_kernel':self._log_kernel(query,source),'log_band_kernels':[bands],
            'band_weights':[per_band],'view_weights':[weights],
            'geometry':f'2D anchored positions and bounded {self.semantic_dim}D learned query/source coordinates',
            'views':1,'semantic_source_norm_compensated':True,'query_row_constant_removed':True,
            'content_length_factor':self.content_length_factor,'content_temperature_factor':self.content_length_factor**2}

class RoutingEncoderBlock(OLD.KernelEncoderBlock):
    def __init__(self,variant,dim,heads=3,semantic_dim=6,content_length_factor=None,**options):
        nn.Module.__init__(self);self.norm1,self.norm2=nn.LayerNorm(dim),nn.LayerNorm(dim)
        self.mlp=nn.Sequential(nn.Linear(dim,4*dim),nn.GELU(),nn.Linear(4*dim,dim))
        if variant=='normalized_kernel':
            if semantic_dim!=6 or content_length_factor not in(None,1,1.):raise ValueError('Exact normalized6D control requires factor1')
            self.mixer=PREVIOUS.GeometryMixer(dim,heads,variant,coordinate_dim=8,**options)
        else:self.mixer=RoutingMixer(dim,heads,variant,semantic_dim=semantic_dim,content_length_factor=content_length_factor,**options)

class KernelVisionEncoder(OLD.KernelVisionEncoder):
    def __init__(self,kind='compensated_kernel',patch_size=8,dim=192,depth=4,heads=3,
                 semantic_dim=6,bandwidths=(.15,.35,.75),content_scale=.5,checkpoint=False,content_length_factor=None):
        if kind not in VARIANTS[1:] or min(patch_size,dim,depth)<1:raise ValueError('Explicit variant and positive model dimensions required')
        nn.Module.__init__(self);self.patch_size,self.dim,self.kind=int(patch_size),int(dim),kind
        self.checkpoint_blocks=bool(checkpoint);self.patch_embed=nn.Linear(3*patch_size*patch_size,dim)
        self.blocks=nn.ModuleList([RoutingEncoderBlock(kind,dim,heads,semantic_dim,content_length_factor,
            bandwidths=bandwidths,content_scale=content_scale) for _ in range(depth)])
        self.norm=nn.LayerNorm(dim)

class KernelJEPA(JEPA):
    def __init__(self,config,variant=None):
        nn.Module.__init__(self);self.config=copy.deepcopy(config);options=config.get('kernel14',{})
        variant=variant or options.get('variant','compensated_kernel');semantic_dim=options.get('semantic_dim',32 if variant=='compensated_wide_kernel'else 6)
        if 'coordinate_dim' in options and int(options['coordinate_dim'])!=int(semantic_dim)+2:raise ValueError('coordinate_dim must equal semantic_dim+2')
        self.context=KernelVisionEncoder(kind=variant,patch_size=config.get('patch_size',8),dim=config.get('dim',192),
            depth=options.get('depth',config.get('depth',4)),heads=options.get('heads',config.get('heads',3)),
            semantic_dim=semantic_dim,bandwidths=options.get('bandwidths',(.15,.35,.75)),content_scale=options.get('content_scale',.5),
            checkpoint=options.get('checkpoint',False),content_length_factor=options.get('content_length_factor'))
        self.target=copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor=Predictor(self.context.dim,dim=config.get('predictor_dim',128),depth=config.get('predictor_depth',2),heads=config.get('predictor_heads',4))

def build_model(config,kind='semantic',variant=None):
    variant=variant or config.get('kernel14',{}).get('variant','compensated_kernel')
    if variant not in VARIANTS:raise ValueError('Explicit registered routing variant required')
    if kind=='vit' or variant=='vit':
        cfg=copy.deepcopy(config);cfg['kind']='vit';return JEPA(cfg)
    if kind!='semantic':raise ValueError('kind must be vit or semantic')
    return KernelJEPA(config,variant)

@torch.no_grad()
def initialize_mixer_from_reference(destination,reference):
    """Copy common6 coordinates per head; retain deterministic equal-variance extra coordinates.
    Wide extra coordinates must come from deterministic destination initialization.
    The new query/key rows start equal but use independent trainable storage.
    """
    if destination.heads!=reference.heads or destination.dim!=reference.dim:raise ValueError('Mixer dimensions differ')
    if reference.coordinate_dim!=8:raise ValueError('Reference must have six semantic coordinates')
    names=[]
    for name in ('value','projection'):
        getattr(destination,name).load_state_dict(getattr(reference,name).state_dict());names.extend(name+'.'+key for key in getattr(destination,name).state_dict())
    for name in ('log_bandwidths','mixture_logits'):
        getattr(destination,name).copy_(getattr(reference,name));names.append(name)
    heads=destination.heads;dest_dim=destination.coordinate_dim-2
    for name in ('query_content','key_content','query_position','key_position'):
        dest_module=getattr(destination,name);ref_module=getattr(reference,name)
        for key,param in dest_module.named_parameters():
            ref_param=dict(ref_module.named_parameters())[key]
            dest=param.reshape(heads,dest_dim,*param.shape[1:]);ref=ref_param.reshape(heads,6,*ref_param.shape[1:])
            dest[:,:6].copy_(ref);names.append(name+'.'+key+('[:,:6 per head]'if dest_dim>6 else''))
    if dest_dim>6:
        for suffix in ('content','position'):
            query=getattr(destination,'query_'+suffix);key=getattr(destination,'key_'+suffix)
            for name,param in key.named_parameters():
                qparam=dict(query.named_parameters())[name]
                param.reshape(heads,dest_dim,*param.shape[1:])[:,6:].copy_(qparam.reshape(heads,dest_dim,*qparam.shape[1:])[:,6:])
    return names

def parameter_counts(model):
    result=OLD.parameter_counts(model);kind=model.context.kind
    result.update(semantic_dimension=64 if kind=='vit'else(32 if kind=='compensated_wide_kernel'else 6),
        parameter_matching='Shared scaffold/values and first6 coordinate maps; extra26 coordinates/head active and counted',
        dense_pair_complexity='O(B H N² S); exact dense kernels, no FMM speedup')
    return result

@torch.no_grad()
def kernel_geometry_diagnostics(model_or_encoder,images,indices=None,valid=None,max_samples=2,**options):
    result=PREVIOUS.kernel_geometry_diagnostics(model_or_encoder,images,indices,valid,max_samples)
    if result.get('status')!='MEASURED':return result
    encoder=getattr(model_or_encoder,'context',model_or_encoder)
    for row,block in zip(result['layers'],encoder.blocks):
        mixer=block.mixer;factor=getattr(mixer,'content_length_factor',1.)
        row.update(semantic_dimension=mixer.coordinate_dim-2,semantic_source_norm_compensated=encoder.kind in VARIANTS[2:],
            content_length_factor=factor,content_temperature_factor=factor**2,
            content_bandwidths=(mixer.log_bandwidths.clamp(math.log(.03),math.log(2.)).exp()*factor).cpu().tolist())
    result['interpretation']='Descriptive exact dense routing geometry. Six and32 semantic coordinates compare capacity and source-norm compensation; no manifold isomorphism or FMM acceleration established.'
    return result

def compensated_gaussian_attention(query,key,value,query_valid,source_valid,temperature=None):
    """Single Gaussian+source-norm compensation equals ordinary dot-product attention.
    Operates on already projected raw Q/K/V[B,H,N,D], with no spatial factor/tanh.
    Dropping the exact query-row constant gives stable logits QK^T/temperature.
    This equivalence helper is not an extra scientific training condition.
    """
    if query.ndim!=4 or key.ndim!=4 or value.ndim!=4 or query.shape[:2]!=key.shape[:2] or key.shape[:3]!=value.shape[:3] or query.shape[-1]!=key.shape[-1]:raise ValueError('Expected compatible Q/K/V[B,H,N,D]')
    if query_valid.shape!=(query.shape[0],query.shape[2]) or source_valid.shape!=(key.shape[0],key.shape[2]):raise ValueError('Validity masks must match token axes')
    if not bool(source_valid.any(-1).all()):raise ValueError('Each sample requires a source')
    temperature=math.sqrt(query.shape[-1])if temperature is None else float(temperature)
    if not math.isfinite(temperature)or temperature<=0:raise ValueError('Positive temperature required')
    q=torch.where(query_valid[:,None,:,None],query,torch.zeros_like(query))
    k=torch.where(source_valid[:,None,:,None],key,torch.zeros_like(key));v=torch.where(source_valid[:,None,:,None],value,torch.zeros_like(value))
    logits=(q@k.transpose(-1,-2))/temperature
    weights=logits.masked_fill(~source_valid[:,None,None,:],float('-inf')).softmax(-1)
    return torch.where(query_valid[:,None,:,None],weights@v,torch.zeros_like(weights@v))
